# 02 - Mainnet CPMM

Raydium CPMM analysis for snapshot-derived CPMM scenarios and the historical CPMM decoding pipeline.

Inputs:
- `results/real_pool_comparison.csv`
- `results/historical_cpmm_swaps_status.csv`
- `results/historical_cpmm_pipeline_summary.csv`
- `results/historical_cpmm_decoded.csv`
- `results/historical_cpmm_candidates.csv`


## Data readiness


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_candidate_summary,
    historical_pipeline_funnel,
    historical_rejection_pareto,
    historical_slippage_summary,
    historical_takeaway_lines,
    historical_top_candidates,
    hypothesis_scorecard,
    load_inputs,
    plot_profit_distribution,
    plot_rejection_pareto,
    plot_size_vs_profit,
    plot_slippage_feasibility,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

comparison = inputs["frames"]["real_pool_comparison"]
historical = inputs["frames"]["historical_cpmm_candidates"]
historical_frames = {
    name: inputs["frames"][name]
    for name in [
        "historical_cpmm_swaps_status",
        "historical_cpmm_pipeline_summary",
        "historical_cpmm_decoded",
        "historical_cpmm_candidates",
    ]
}
funnel = historical_pipeline_funnel(historical_frames)
rejections = historical_rejection_pareto(historical_frames)

readiness_names = ["real_pool_comparison", *historical_frames.keys()]
display(data_readiness(ROOT, inputs, readiness_names))
if comparison.attrs.get("legacy_schema", False):
    display(Markdown("> CPMM comparison CSV is old-schema. Regenerate before final thesis numbers."))


## Snapshot CPMM scorecard


In [ ]:
if comparison.empty:
    display(Markdown("Generate `results/real_pool_comparison.csv` first."))
else:
    display(hypothesis_scorecard(comparison))
    display(blocker_table(comparison))


## Historical pipeline funnel


In [ ]:
display(funnel)


## Rejection Pareto


In [ ]:
if rejections.empty:
    display(Markdown("No historical CPMM rejection reasons available yet."))
else:
    display(rejections)
    fig = plot_rejection_pareto(rejections)
    if fig is None:
        display(Markdown("Rejection Pareto plot omitted because the data is flat or single-point."))


## Historical candidate summary


In [ ]:
display(historical_candidate_summary(historical))


## Profit distribution


In [ ]:
fig = plot_profit_distribution(historical)
if fig is None:
    display(Markdown("Profit distribution plot omitted because candidate profit data is missing, flat, or single-point."))


## Size vs profit


In [ ]:
fig = plot_size_vs_profit(historical)
if fig is None:
    display(Markdown("Size-vs-profit plot omitted because candidate size/profit data is missing, flat, or single-point."))


## Slippage feasibility


In [ ]:
slippage_summary = historical_slippage_summary(historical)
if slippage_summary.empty:
    display(Markdown("No historical CPMM slippage feasibility columns available yet."))
else:
    display(slippage_summary)
fig = plot_slippage_feasibility(historical)
if fig is None:
    display(Markdown("Slippage feasibility plot omitted because the data is missing, flat, or single-point."))


## Top-N candidates


In [ ]:
top_candidates = historical_top_candidates(historical, limit=10)
if top_candidates.empty:
    display(Markdown("No ranked historical CPMM candidates yet."))
else:
    display(top_candidates)


## Thesis-ready takeaways


In [ ]:
lines = historical_takeaway_lines(comparison, historical, funnel, rejections)
display(Markdown("\n".join(lines) if lines else "No CPMM conclusions yet."))
